In [ ]:
# Cell 1 — Preflight only: find the saved synthetic inputs and confirm GPU/JAX.
from pathlib import Path
import json, shutil, subprocess, sys
import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
VALIDATION_CANDIDATES = []
for folder in INPUT.rglob("nuts_validation_inputs"):
    if folder.is_dir() and all((folder / name).is_file() for name in
                               ("binary.csv", "zeeman.csv", "known_truth.json")):
        VALIDATION_CANDIDATES.append(folder)

if not VALIDATION_CANDIDATES:
    raise FileNotFoundError(
        "Attach the saved Day 7 Physics notebook output as Kaggle Input. "
        "It must contain nuts_validation_inputs/binary.csv, zeeman.csv, and known_truth.json."
    )

# Prefer the most recently modified complete validation folder; print all candidates for audit.
VALIDATION_CANDIDATES.sort(key=lambda p: p.stat().st_mtime, reverse=True)
VALIDATION_DIR = VALIDATION_CANDIDATES[0]
print("Validation input selected:", VALIDATION_DIR)
print("Complete validation folders found:", len(VALIDATION_CANDIDATES))

try:
    import jax
    import numpyro
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpyro>=0.15"], check=True)
    import jax
    import numpyro

if jax.default_backend() != "gpu":
    raise RuntimeError(
        f"JAX backend is {jax.default_backend()}, not GPU. Set Kaggle Accelerator to GPU T4 x2, "
        "restart the session, then Save Version and Run All."
    )
DEVICES = jax.devices()
print("JAX backend:", jax.default_backend(), "| devices:", DEVICES)
print("NumPyro:", numpyro.__version__)

binary = pd.read_csv(VALIDATION_DIR / "binary.csv")
zeeman = pd.read_csv(VALIDATION_DIR / "zeeman.csv")
TRUTH = json.loads((VALIDATION_DIR / "known_truth.json").read_text(encoding="utf-8"))
required_binary = {"time_days", "flux", "flux_err"}
required_zeeman = {"wavelength_angstrom", "normalized_flux", "flux_err"}
assert required_binary.issubset(binary.columns), f"Binary columns missing: {required_binary - set(binary.columns)}"
assert required_zeeman.issubset(zeeman.columns), f"Zeeman columns missing: {required_zeeman - set(zeeman.columns)}"
assert len(binary) >= 100 and len(zeeman) >= 100, "Synthetic validation inputs appear incomplete."

OUT = WORK / "day8-targeted-bayesian"
OUT.mkdir(parents=True, exist_ok=True)
print("[PASS] Binary rows:", len(binary), "| Zeeman rows:", len(zeeman))
print("[PASS] Output directory:", OUT)
print("Preflight passed. No sampling has started in this cell.")


In [ ]:
# Cell 2 — Run the two reduced, identifiable synthetic NUTS recoveries.
import time
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary
from astropy.timeseries import BoxLeastSquares

SEED = 815
WARMUP = 400
SAMPLES = 600
CHAINS = 2
CHAIN_METHOD = "parallel" if len(DEVICES) >= CHAINS else "sequential"
TARGET_ACCEPT = 0.92
MAX_TREE_DEPTH = 9
FIT_RESULTS = {}

def run_fit(name, model, init_values, seed):
    started = time.time()
    kernel = NUTS(model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH,
                  init_strategy=init_to_value(values=init_values))
    mcmc = MCMC(kernel, num_warmup=WARMUP, num_samples=SAMPLES,
                num_chains=CHAINS, chain_method=CHAIN_METHOD, progress_bar=True)
    mcmc.run(jax.random.PRNGKey(seed))
    grouped = mcmc.get_samples(group_by_chain=True)
    flat = mcmc.get_samples(group_by_chain=False)
    stats = numpyro_summary(grouped, group_by_chain=True)
    rhat = [float(np.asarray(v["r_hat"])) for v in stats.values()
            if "r_hat" in v and np.asarray(v["r_hat"]).ndim == 0 and np.isfinite(v["r_hat"])]
    ess = [float(np.asarray(v["n_eff"])) for v in stats.values()
           if "n_eff" in v and np.asarray(v["n_eff"]).ndim == 0 and np.isfinite(v["n_eff"])]
    extra = mcmc.get_extra_fields(group_by_chain=True)
    divergences = int(np.asarray(extra.get("diverging", [])).sum())
    max_rhat = max(rhat) if rhat else None
    min_ess = min(ess) if ess else None
    passed = (max_rhat is not None and max_rhat <= 1.05 and
              min_ess is not None and min_ess >= 100 and divergences == 0)
    result = {
        "family": name, "status": "pilot_diagnostics_pass" if passed else "needs_review",
        "seconds": round(time.time() - started, 1), "chains": CHAINS,
        "warmup_per_chain": WARMUP, "samples_per_chain": SAMPLES,
        "chain_method": CHAIN_METHOD, "max_r_hat": max_rhat,
        "min_effective_sample_size": min_ess, "divergences": divergences,
        "screening_thresholds": {"max_r_hat": 1.05, "min_effective_sample_size": 100, "divergences": 0},
        "samples": {k: np.asarray(v) for k, v in flat.items()},
    }
    FIT_RESULTS[name] = result
    print(f"[{result['status']}] {name}: {result['seconds']} sec; "
          f"max R-hat={max_rhat}; min ESS={min_ess}; divergences={divergences}", flush=True)
    return result

def try_fit(name, model, init_values, seed):
    try:
        return run_fit(name, model, init_values, seed)
    except Exception as exc:
        FIT_RESULTS[name] = {
            "family": name, "status": "failed",
            "error": f"{type(exc).__name__}: {exc}",
        }
        print(f"[FAILED] {name}: {type(exc).__name__}: {exc}", flush=True)
        return FIT_RESULTS[name]

# Binary data preparation and a BLS local mode estimate.
bt = binary["time_days"].to_numpy(float)
bf = binary["flux"].to_numpy(float)
be = binary["flux_err"].to_numpy(float)
valid = np.isfinite(bt) & np.isfinite(bf) & np.isfinite(be) & (be > 0)
bt, bf, be = bt[valid], bf[valid], be[valid]
order = np.argsort(bt)
bt, bf, be = bt[order], bf[order], be[order]
bmed = np.median(bf)
bf, be = bf / bmed, be / bmed
bt = bt - bt.min()
bmax_period = min(13.0, np.ptp(bt) / 2.0)
durations = np.geomspace(0.005, min(0.09, bmax_period * 0.12), 14)
bls = BoxLeastSquares(bt, bf, dy=be).autopower(
    duration=durations, objective="snr", minimum_period=0.1,
    maximum_period=bmax_period, frequency_factor=20.0,
)
best = int(np.nanargmax(bls.power))
BLS_PERIOD = float(bls.period[best])
BLS_PHASE = float(np.mod(bls.transit_time[best] / BLS_PERIOD, 1.0))
BLS_DEPTH = float(np.clip(bls.depth[best], 0.03, 0.35))
BLS_DURATION = float(np.clip(bls.duration[best] / BLS_PERIOD, 0.02, 0.12))
print(f"Binary BLS local mode: period={BLS_PERIOD:.6f} d, phase={BLS_PHASE:.5f}, "
      f"duration_fraction={BLS_DURATION:.5f}", flush=True)

bt_j, bf_j, be_j = map(jnp.asarray, (bt, bf, be))
def binary_model():
    period = numpyro.sample(
        "period_days",
        dist.TruncatedNormal(BLS_PERIOD, 0.012, low=max(0.1, BLS_PERIOD - 0.06), high=BLS_PERIOD + 0.06),
    )
    phase_offset = numpyro.sample("phase_offset", dist.Normal(0.0, 0.025))
    phase = BLS_PHASE + phase_offset
    depth = numpyro.sample("depth_primary", dist.Uniform(0.03, 0.35))
    depth_ratio = numpyro.sample("depth_ratio", dist.Uniform(0.0, 1.0))
    duration_fraction = numpyro.sample("duration_fraction", dist.Uniform(0.02, 0.12))
    ellipsoidal = numpyro.sample("ellipsoidal_amplitude", dist.Normal(0.0, 0.01))
    reflection = numpyro.sample("reflection_amplitude", dist.Normal(0.0, 0.01))
    extra_noise = numpyro.sample("extra_noise", dist.HalfNormal(0.01))
    orbital = jnp.mod(bt_j / period - phase + 0.5, 1.0) - 0.5
    half = duration_fraction / 2.0
    ingress = half * 0.18
    primary = jnp.clip((half + ingress - jnp.abs(orbital)) / (2.0 * ingress), 0.0, 1.0)
    secondary_distance = 0.5 - jnp.abs(orbital)
    secondary = jnp.clip((half + ingress - jnp.abs(secondary_distance)) / (2.0 * ingress), 0.0, 1.0)
    prediction = 1.0 - depth * primary - depth * depth_ratio * secondary
    prediction += ellipsoidal * jnp.cos(4.0 * jnp.pi * orbital)
    prediction += reflection * jnp.cos(2.0 * jnp.pi * orbital)
    sigma = jnp.sqrt(be_j**2 + extra_noise**2)
    numpyro.sample("flux", dist.Normal(prediction, sigma), obs=bf_j)

binary_init = {
    "period_days": BLS_PERIOD, "phase_offset": 0.0, "depth_primary": BLS_DEPTH,
    "depth_ratio": 0.45, "duration_fraction": BLS_DURATION,
    "ellipsoidal_amplitude": 0.0, "reflection_amplitude": 0.0, "extra_noise": 0.002,
}
try_fit("binary", binary_model, binary_init, SEED)

# Zeeman data: use only the local H-alpha window. Rest wavelength is fixed from
# line identification, removing the center-vs-velocity degeneracy.
REST_WAVELENGTH = 6562.8
zc = zeeman["wavelength_angstrom"].to_numpy(float)
zf = zeeman["normalized_flux"].to_numpy(float)
ze = zeeman["flux_err"].to_numpy(float)
zvalid = np.isfinite(zc) & np.isfinite(zf) & np.isfinite(ze) & (ze > 0)
zc, zf, ze = zc[zvalid], zf[zvalid], ze[zvalid]
window = np.abs(zc - REST_WAVELENGTH) <= 8.0
zc, zf, ze = zc[window], zf[window], ze[window]
if len(zc) < 50:
    raise RuntimeError(f"Only {len(zc)} samples in the H-alpha window; expected at least 50.")
zc_j, zf_j, ze_j = map(jnp.asarray, (zc, zf, ze))
def zeeman_model():
    field = numpyro.sample("field_gauss", dist.Uniform(0.0, 50_000.0))
    velocity = numpyro.sample("radial_velocity_km_s", dist.Normal(0.0, 100.0))
    depth = numpyro.sample("line_depth", dist.Uniform(0.1, 0.85))
    width = numpyro.sample("line_sigma_angstrom", dist.Uniform(0.25, 1.5))
    continuum = numpyro.sample("continuum_offset", dist.Normal(0.0, 0.01))
    intrinsic_noise = numpyro.sample("intrinsic_noise", dist.HalfNormal(0.01))
    shifted = REST_WAVELENGTH * (1.0 + velocity / 299792.458)
    delta = 4.67e-13 * shifted**2 * field
    pi = jnp.exp(-0.5 * ((zc_j - shifted) / width) ** 2)
    blue = jnp.exp(-0.5 * ((zc_j - (shifted - delta)) / width) ** 2)
    red = jnp.exp(-0.5 * ((zc_j - (shifted + delta)) / width) ** 2)
    prediction = 1.0 + continuum - depth * (pi + blue + red) / 3.0
    sigma = jnp.sqrt(ze_j**2 + intrinsic_noise**2)
    numpyro.sample("spectrum", dist.Normal(prediction, sigma), obs=zf_j)

zeeman_init = {
    "field_gauss": 15_000.0, "radial_velocity_km_s": 0.0,
    "line_depth": 0.6, "line_sigma_angstrom": 0.5,
    "continuum_offset": 0.0, "intrinsic_noise": 0.002,
}
print("Zeeman local-window samples:", len(zc), "| fixed H-alpha rest wavelength:", REST_WAVELENGTH, flush=True)
try_fit("zeeman", zeeman_model, zeeman_init, SEED + 1)

print("Sampling cell finished. Review per-family status before interpreting intervals.")


In [ ]:
# Cell 3 — Summarize recovery against withheld synthetic truth, plot fits, and save the report.
import json
import matplotlib.pyplot as plt

def interval_record(family, parameter, samples, truth_value=None, units=""):
    values = np.asarray(samples, dtype=float)
    q05, med, q95 = np.quantile(values, [0.05, 0.50, 0.95])
    record = {
        "family": family, "parameter": parameter, "unit": units,
        "q05": float(q05), "median": float(med), "q95": float(q95),
        "known_truth": None if truth_value is None else float(truth_value),
        "truth_inside_90pct_interval": None if truth_value is None else bool(q05 <= truth_value <= q95),
    }
    return record

records = []
truth_binary = TRUTH.get("binary", {})
truth_zeeman = TRUTH.get("zeeman", {})
if "binary" in FIT_RESULTS and "samples" in FIT_RESULTS["binary"]:
    s = FIT_RESULTS["binary"]["samples"]
    records += [
        interval_record("binary", "period_days", s["period_days"], truth_binary.get("period_days"), "day"),
        interval_record("binary", "phase", np.mod(BLS_PHASE + s["phase_offset"], 1.0), truth_binary.get("phase"), "cycle"),
        interval_record("binary", "depth_primary", s["depth_primary"], truth_binary.get("depth_primary")),
        interval_record("binary", "depth_ratio", s["depth_ratio"], truth_binary.get("depth_ratio")),
        interval_record("binary", "duration_fraction", s["duration_fraction"], truth_binary.get("duration_fraction"), "cycle"),
        interval_record("binary", "extra_noise", s["extra_noise"], truth_binary.get("extra_noise")),
    ]
if "zeeman" in FIT_RESULTS and "samples" in FIT_RESULTS["zeeman"]:
    s = FIT_RESULTS["zeeman"]["samples"]
    records += [
        interval_record("zeeman", "field_gauss", s["field_gauss"], truth_zeeman.get("field_gauss"), "gauss"),
        interval_record("zeeman", "radial_velocity_km_s", s["radial_velocity_km_s"], truth_zeeman.get("radial_velocity_km_s"), "km/s"),
        interval_record("zeeman", "line_depth", s["line_depth"], truth_zeeman.get("line_depth")),
        interval_record("zeeman", "line_sigma_angstrom", s["line_sigma_angstrom"], truth_zeeman.get("line_sigma_angstrom"), "angstrom"),
        interval_record("zeeman", "intrinsic_noise", s["intrinsic_noise"], truth_zeeman.get("intrinsic_noise")),
    ]

pd.DataFrame(records).to_csv(OUT / "day8_posterior_recovery_intervals.csv", index=False)

def binary_curve(time, period, phase, depth, ratio, duration, ellip, refl):
    orbital = np.mod(time / period - phase + 0.5, 1.0) - 0.5
    half = duration / 2.0
    ingress = half * 0.18
    primary = np.clip((half + ingress - np.abs(orbital)) / (2.0 * ingress), 0.0, 1.0)
    secondary_distance = 0.5 - np.abs(orbital)
    secondary = np.clip((half + ingress - np.abs(secondary_distance)) / (2.0 * ingress), 0.0, 1.0)
    return (1.0 - depth * primary - depth * ratio * secondary
            + ellip * np.cos(4*np.pi*orbital) + refl * np.cos(2*np.pi*orbital))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if "binary" in FIT_RESULTS and "samples" in FIT_RESULTS["binary"]:
    s = FIT_RESULTS["binary"]["samples"]
    med = {k: float(np.median(v)) for k, v in s.items()}
    tgrid = np.linspace(bt.min(), bt.max(), 5000)
    phase_med = float(np.mod(BLS_PHASE + med["phase_offset"], 1.0))
    axes[0].scatter(bt, bf, s=5, alpha=0.35, label="synthetic data")
    axes[0].plot(tgrid, binary_curve(tgrid, med["period_days"], phase_med,
                    med["depth_primary"], med["depth_ratio"], med["duration_fraction"],
                    med["ellipsoidal_amplitude"], med["reflection_amplitude"]),
                 color="crimson", lw=1.2, label="posterior-median model")
    axes[0].set(title="Binary: local BLS mode recovery", xlabel="Time (days)", ylabel="Normalized flux")
    axes[0].legend()
else:
    axes[0].set_visible(False)

if "zeeman" in FIT_RESULTS and "samples" in FIT_RESULTS["zeeman"]:
    s = FIT_RESULTS["zeeman"]["samples"]
    med = {k: float(np.median(v)) for k, v in s.items()}
    wgrid = np.linspace(zc.min(), zc.max(), 1200)
    shifted = REST_WAVELENGTH * (1.0 + med["radial_velocity_km_s"] / 299792.458)
    delta = 4.67e-13 * shifted**2 * med["field_gauss"]
    profile = (np.exp(-0.5*((wgrid-shifted)/med["line_sigma_angstrom"])**2)
               + np.exp(-0.5*((wgrid-(shifted-delta))/med["line_sigma_angstrom"])**2)
               + np.exp(-0.5*((wgrid-(shifted+delta))/med["line_sigma_angstrom"])**2)) / 3.0
    pred = 1.0 + med["continuum_offset"] - med["line_depth"]*profile
    axes[1].scatter(zc, zf, s=8, alpha=0.45, label="synthetic data")
    axes[1].plot(wgrid, pred, color="crimson", lw=1.5, label="posterior-median model")
    axes[1].set(title="Zeeman: H-alpha conditional recovery", xlabel="Wavelength (Å)", ylabel="Normalized flux")
    axes[1].legend()
else:
    axes[1].set_visible(False)

fig.tight_layout()
fig.savefig(OUT / "day8_posterior_median_fits.png", dpi=160)
plt.show()

report = {
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "validation_input_dir": str(VALIDATION_DIR),
    "jax_backend": jax.default_backend(),
    "jax_devices": [str(x) for x in DEVICES],
    "scope": "Conditional synthetic recovery; binary period is localized around a BLS mode; Zeeman H-alpha rest wavelength is fixed.",
    "results": {
        family: {k: v for k, v in result.items() if k != "samples"}
        for family, result in FIT_RESULTS.items()
    },
    "posterior_interval_file": str(OUT / "day8_posterior_recovery_intervals.csv"),
    "fit_plot": str(OUT / "day8_posterior_median_fits.png"),
    "interpretation_limits": [
        "These are known-truth synthetic validation examples, not posterior claims for real TESS candidates.",
        "A BLS-localized binary fit does not establish global period recovery or resolve all aliases.",
        "Fixing the H-alpha rest wavelength uses an identified-line assumption and does not validate automated line identification.",
        "Two-chain diagnostics are a quick screening check; promote neither model as validated if it is marked needs_review.",
        "The earlier flow posterior undercoverage remains a separate issue and is not repaired by this NUTS run."
    ]
}
(OUT / "day8_targeted_bayesian_summary.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print(pd.DataFrame([{k: r.get(k) for k in ("family", "status", "max_r_hat", "min_effective_sample_size", "divergences", "seconds")}
                    for r in report["results"].values()]).to_string(index=False))
print("Saved posterior intervals:", OUT / "day8_posterior_recovery_intervals.csv")
print("Saved fit plot:", OUT / "day8_posterior_median_fits.png")
print("Saved summary:", OUT / "day8_targeted_bayesian_summary.json")
print("Kaggle output files are under:", OUT)
